In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Load your processed dataset**

In [2]:
import os
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/bigdata-gbl-quiz"
PROCESSED_FILE = os.path.join(PROJECT_ROOT, "data", "processed", "quiz_features.csv")

df = pd.read_csv(PROCESSED_FILE)
print("Shape:", df.shape)
df.head()

Shape: (19991, 15)


,learner_id,question_id,topic,difficulty,time_spent_seconds,attempts_count,hints_used,correct,time_per_attempt,attempt_index,recent_success_5,recent_hints_5,recent_time_5,questions_completed,avg_difficulty_so_far
0,L0001,Q0103,Geography,2,73,3,2,0,24.333333,1,0.500000,0.0,56.000000,1,2.000000
1,L0001,Q0561,History,2,74,3,1,0,24.666667,2,0.000000,2.0,73.000000,2,2.000000
2,L0001,Q0189,ArtsCulture,1,37,2,0,1,18.500000,3,0.000000,1.5,73.500000,3,2.000000
3,L0001,Q0443,ArtsCulture,1,60,2,1,0,30.000000,4,0.333333,1.0,61.333333,4,1.666667
4,L0001,Q0392,Technology,1,33,2,0,1,16.500000,5,0.250000,1.0,61.000000,5,1.500000


**Rebuild your Logistic Regression model (fast + reliable)**

In [3]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

target = "correct"
feature_cols = [
    "difficulty","time_spent_seconds","attempts_count","hints_used","time_per_attempt",
    "recent_success_5","recent_hints_5","recent_time_5","avg_difficulty_so_far",
    "questions_completed","topic"
]

df_model = df[["learner_id"] + feature_cols + [target]].copy()

X = df_model[feature_cols]
y = df_model[target]
groups = df_model["learner_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, y_train = X.iloc[train_idx].copy(), y.iloc[train_idx].copy()

numeric_features = [c for c in feature_cols if c != "topic"]
categorical_features = ["topic"]

preprocess_lr = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

lr_model = Pipeline(steps=[
    ("preprocess", preprocess_lr),
    ("model", LogisticRegression(max_iter=300, class_weight="balanced"))
])

lr_model.fit(X_train, y_train)
print("✅ Logistic Regression model trained")

✅ Logistic Regression model trained


**Add predicted probability (p_correct) to each row**

In [4]:
df_model["p_correct"] = lr_model.predict_proba(X)[:, 1]
df_model[["learner_id","topic","difficulty","p_correct","correct"]].head(10)

,learner_id,topic,difficulty,p_correct,correct
0,L0001,Geography,2,0.561953,0
1,L0001,History,2,0.462899,0
2,L0001,ArtsCulture,1,0.569461,1
3,L0001,ArtsCulture,1,0.588446,0
4,L0001,Technology,1,0.606290,1
5,L0001,Sports,1,0.745364,1
6,L0001,History,1,0.754437,0
7,L0001,Technology,1,0.754226,0
8,L0001,History,2,0.622299,0
9,L0001,Geography,1,0.705641,1


**Create the recommendation function (easy / same / harder)**

In [5]:
def recommend_next_difficulty(current_difficulty, p_correct):
    if p_correct < 0.35:
        return max(1, current_difficulty - 1), "Likely to struggle → easier + support"
    elif p_correct > 0.70:
        return min(3, current_difficulty + 1), "Likely to succeed → harder challenge"
    else:
        return current_difficulty, "Balanced zone → keep difficulty"

df_model[["rec_difficulty", "rec_reason"]] = df_model.apply(
    lambda r: pd.Series(recommend_next_difficulty(int(r["difficulty"]), float(r["p_correct"]))),
    axis=1
)

df_model[["difficulty","p_correct","rec_difficulty","rec_reason"]].head(12)

,difficulty,p_correct,rec_difficulty,rec_reason
0,2,0.561953,2,Balanced zone → keep difficulty
1,2,0.462899,2,Balanced zone → keep difficulty
2,1,0.569461,1,Balanced zone → keep difficulty
3,1,0.588446,1,Balanced zone → keep difficulty
4,1,0.606290,1,Balanced zone → keep difficulty
5,1,0.745364,2,Likely to succeed → harder challenge
6,1,0.754437,2,Likely to succeed → harder challenge
7,1,0.754226,2,Likely to succeed → harder challenge
8,2,0.622299,2,Balanced zone → keep difficulty
9,1,0.705641,2,Likely to succeed → harder challenge


**Show 1 learner “personalised journey” (case study)**

In [6]:
learner = df_model["learner_id"].drop_duplicates().sample(1, random_state=42).iloc[0]
learner

'L0204'

In [7]:
case = df_model[df_model["learner_id"] == learner].head(10)[
    ["topic","difficulty","recent_success_5","hints_used","attempts_count","p_correct","rec_difficulty","rec_reason","correct"]
]
case

,topic,difficulty,recent_success_5,hints_used,attempts_count,p_correct,rec_difficulty,rec_reason,correct
13597,Geography,3,0.500000,1,4,0.324313,2,Likely to struggle → easier + support,1
13598,Technology,3,1.000000,2,4,0.453031,3,Balanced zone → keep difficulty,0
13599,ArtsCulture,2,0.500000,0,2,0.560861,2,Balanced zone → keep difficulty,1
13600,Technology,2,0.666667,1,2,0.638048,2,Balanced zone → keep difficulty,0
13601,History,1,0.500000,0,2,0.660371,1,Balanced zone → keep difficulty,0
13602,ArtsCulture,3,0.400000,1,3,0.439056,3,Balanced zone → keep difficulty,0
13603,Technology,2,0.200000,2,2,0.630013,2,Balanced zone → keep difficulty,0
13604,Science,2,0.200000,1,3,0.451148,2,Balanced zone → keep difficulty,1
13605,Technology,1,0.200000,2,1,0.814121,2,Likely to succeed → harder challenge,1
13606,Geography,1,0.400000,0,2,0.657670,1,Balanced zone → keep difficulty,1


**Create “topic weakness” per learner**

In [8]:
# learner performance by topic
topic_perf = (
    df_model.groupby(["learner_id", "topic"])["correct"]
    .mean()
    .reset_index(name="topic_accuracy")
)

topic_perf.head()

,learner_id,topic,topic_accuracy
0,L0001,ArtsCulture,0.416667
1,L0001,Geography,0.181818
2,L0001,History,0.300000
3,L0001,Science,0.250000
4,L0001,Sports,0.833333


**Find each learner’s weakest topic**

In [9]:
weakest_topic = (
    topic_perf.sort_values(["learner_id", "topic_accuracy"])
    .groupby("learner_id")
    .head(1)
    .rename(columns={"topic": "weak_topic"})
    [["learner_id", "weak_topic", "topic_accuracy"]]
)

weakest_topic.head()

,learner_id,weak_topic,topic_accuracy
1,L0001,Geography,0.181818
11,L0002,Technology,0.000000
16,L0003,Sports,0.250000
22,L0004,Sports,0.230769
25,L0005,Geography,0.000000


**Merge weak topic back into your main table**

In [11]:
df_model = df_model.merge(weakest_topic, on="learner_id", how="left")
df_model[["learner_id", "weak_topic", "topic_accuracy"]].head()

,learner_id,weak_topic,topic_accuracy
0,L0001,Geography,0.181818
1,L0001,Geography,0.181818
2,L0001,Geography,0.181818
3,L0001,Geography,0.181818
4,L0001,Geography,0.181818


**Create a “next topic” recommendation rule**

In [12]:
def recommend_next_topic(current_topic, weak_topic, p_correct):
    if p_correct < 0.35:
        return weak_topic, "Focus weak area (support learning gaps)"
    elif p_correct > 0.70:
        # move away from weak topic to maintain variety/challenge
        return current_topic, "Maintain momentum (variety/challenge)"
    else:
        return current_topic, "Keep topic stable (balanced zone)"

df_model[["rec_topic", "rec_topic_reason"]] = df_model.apply(
    lambda r: pd.Series(recommend_next_topic(
        r["topic"], r["weak_topic"], float(r["p_correct"])
    )),
    axis=1
)

df_model[["topic","weak_topic","p_correct","rec_topic","rec_topic_reason"]].head(10)

,topic,weak_topic,p_correct,rec_topic,rec_topic_reason
0,Geography,Geography,0.561953,Geography,Keep topic stable (balanced zone)
1,History,Geography,0.462899,History,Keep topic stable (balanced zone)
2,ArtsCulture,Geography,0.569461,ArtsCulture,Keep topic stable (balanced zone)
3,ArtsCulture,Geography,0.588446,ArtsCulture,Keep topic stable (balanced zone)
4,Technology,Geography,0.606290,Technology,Keep topic stable (balanced zone)
5,Sports,Geography,0.745364,Sports,Maintain momentum (variety/challenge)
6,History,Geography,0.754437,History,Maintain momentum (variety/challenge)
7,Technology,Geography,0.754226,Technology,Maintain momentum (variety/challenge)
8,History,Geography,0.622299,History,Keep topic stable (balanced zone)
9,Geography,Geography,0.705641,Geography,Maintain momentum (variety/challenge)


**“support action” recommendation (very realistic)**

In [13]:
def recommend_support(p_correct, hints_used, attempts_count):
    if p_correct < 0.35:
        return "Offer hint early + suggest 2-minute revision card", "Reduce frustration / prevent failure loop"
    elif p_correct > 0.70:
        return "Limit hints + suggest harder question bonus", "Maintain challenge and engagement"
    else:
        # balanced
        if hints_used >= 1 or attempts_count >= 3:
            return "Give gentle hint + brief feedback", "Support without lowering challenge"
        return "Standard feedback", "Keep flow"

df_model[["support_action", "support_reason"]] = df_model.apply(
    lambda r: pd.Series(recommend_support(float(r["p_correct"]), int(r["hints_used"]), int(r["attempts_count"]))),
    axis=1
)

df_model[["p_correct","hints_used","attempts_count","support_action","support_reason"]].head(12)

,p_correct,hints_used,attempts_count,support_action,support_reason
0,0.561953,2,3,Give gentle hint + brief feedback,Support without lowering challenge
1,0.462899,1,3,Give gentle hint + brief feedback,Support without lowering challenge
2,0.569461,0,2,Standard feedback,Keep flow
3,0.588446,1,2,Give gentle hint + brief feedback,Support without lowering challenge
4,0.606290,0,2,Standard feedback,Keep flow
5,0.745364,0,1,Limit hints + suggest harder question bonus,Maintain challenge and engagement
6,0.754437,0,1,Limit hints + suggest harder question bonus,Maintain challenge and engagement
7,0.754226,1,2,Limit hints + suggest harder question bonus,Maintain challenge and engagement
8,0.622299,0,2,Standard feedback,Keep flow
9,0.705641,1,2,Limit hints + suggest harder question bonus,Maintain challenge and engagement


**full adaptive decision**

In [14]:
case2 = df_model[df_model["learner_id"] == learner].head(12)[
    ["topic","difficulty","p_correct",
     "rec_difficulty","rec_reason",
     "rec_topic","rec_topic_reason",
     "support_action",
     "correct"]
]
case2

,topic,difficulty,p_correct,rec_difficulty,rec_reason,rec_topic,rec_topic_reason,support_action,correct
13597,Geography,3,0.324313,2,Likely to struggle → easier + support,Technology,Focus weak area (support learning gaps),Offer hint early + suggest 2-minute revision card,1
13598,Technology,3,0.453031,3,Balanced zone → keep difficulty,Technology,Keep topic stable (balanced zone),Give gentle hint + brief feedback,0
13599,ArtsCulture,2,0.560861,2,Balanced zone → keep difficulty,ArtsCulture,Keep topic stable (balanced zone),Standard feedback,1
13600,Technology,2,0.638048,2,Balanced zone → keep difficulty,Technology,Keep topic stable (balanced zone),Give gentle hint + brief feedback,0
13601,History,1,0.660371,1,Balanced zone → keep difficulty,History,Keep topic stable (balanced zone),Standard feedback,0
13602,ArtsCulture,3,0.439056,3,Balanced zone → keep difficulty,ArtsCulture,Keep topic stable (balanced zone),Give gentle hint + brief feedback,0
13603,Technology,2,0.630013,2,Balanced zone → keep difficulty,Technology,Keep topic stable (balanced zone),Give gentle hint + brief feedback,0
13604,Science,2,0.451148,2,Balanced zone → keep difficulty,Science,Keep topic stable (balanced zone),Give gentle hint + brief feedback,1
13605,Technology,1,0.814121,2,Likely to succeed → harder challenge,Technology,Maintain momentum (variety/challenge),Limit hints + suggest harder question bonus,1
13606,Geography,1,0.657670,1,Balanced zone → keep difficulty,Geography,Keep topic stable (balanced zone),Standard feedback,1


**Save case2 as a CSV in your report/ folder**

In [15]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/bigdata-gbl-quiz"
out_case = os.path.join(PROJECT_ROOT, "report", f"case_study_{learner}.csv")
case2.to_csv(out_case, index=False)

print("✅ Saved case study to:", out_case)

✅ Saved case study to: /content/drive/MyDrive/bigdata-gbl-quiz/report/case_study_L0204.csv
